In [3]:
import sys
sys.path.append("../src")
import cleaning
import pandas as pd

train = pd.read_csv("../data/messy/bank_marketing/outlier_severe/train.csv")
cleaned = cleaning.handle_outlier(train, "y")

print("NaN % per column after outlier step:")
print((cleaned.isnull().mean() * 100).round(2).to_string())

NaN % per column after outlier step:
age               30.22
job                0.00
marital            0.00
education          0.00
default            0.00
housing            0.00
loan               0.00
contact            0.00
month              0.00
day_of_week        0.00
duration          27.82
campaign          25.21
pdays             18.10
previous          26.69
poutcome           0.00
emp.var.rate      31.57
cons.price.idx    25.56
cons.conf.idx     31.02
euribor3m         32.45
nr.employed       25.32
y                  0.00


In [4]:
clean = pd.read_csv("../data/messy/bank_marketing/clean/train.csv")
messy = pd.read_csv("../data/messy/bank_marketing/outlier_severe/train.csv")

flagged_messy = cleaning.handle_outlier(messy, "y").isnull()
flagged_clean = cleaning.handle_outlier(clean, "y").isnull()

rows = []
for col in ["age", "duration", "campaign", "pdays", "previous",
            "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]:
    injected = messy[col] != clean[col]          # truly injected outliers
    flagged = flagged_messy[col]                 # what Isolation Forest flagged

    caught = (flagged & injected).sum() / injected.sum() * 100
    correct = (flagged & injected).sum() / flagged.sum() * 100
    on_clean = flagged_clean[col].mean() * 100

    rows.append([col, round(caught, 1), round(correct, 1), round(on_clean, 1)])

result = pd.DataFrame(rows, columns=["column", "caught_%", "correct_%", "flagged_on_clean_%"])
print(result.to_string(index=False))

        column  caught_%  correct_%  flagged_on_clean_%
           age     100.0       49.6                29.9
      duration     100.0       53.9                19.8
      campaign     100.0       59.4                18.4
         pdays     100.0       82.9                 3.7
      previous     100.0       56.2                13.8
  emp.var.rate     100.0       47.5                38.2
cons.price.idx     100.0       58.7                23.6
 cons.conf.idx     100.0       48.4                18.9
     euribor3m     100.0       46.2                42.8
   nr.employed     100.0       59.2                21.1


In [5]:
from sklearn.ensemble import IsolationForest

num_cols = ["age", "duration", "campaign", "pdays", "previous",
            "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

for name, data in [("clean", clean), ("outlier_severe", messy)]:
    model = IsolationForest(contamination="auto", random_state=42)
    model.fit(data[num_cols])
    pred = model.predict(data[num_cols])
    print(f"{name}: rows flagged = {(pred == -1).mean() * 100:.1f}%")

# how many flagged rows really contain an injected outlier?
has_outlier = (messy[num_cols] != clean[num_cols]).any(axis=1)
model = IsolationForest(contamination="auto", random_state=42)
flagged = model.fit(messy[num_cols]).predict(messy[num_cols]) == -1
print(f"rows with an injected outlier: {has_outlier.mean() * 100:.1f}%")
print(f"correct_% of flagged rows: {(flagged & has_outlier).sum() / flagged.sum() * 100:.1f}")

clean: rows flagged = 18.3%
outlier_severe: rows flagged = 21.5%
rows with an injected outlier: 15.0%
correct_% of flagged rows: 69.6


In [6]:
model = IsolationForest(contamination="auto", random_state=42)
flagged_clean = model.fit(clean[num_cols]).predict(clean[num_cols]) == -1

print(f"y=1 rate in ALL clean rows:     {clean['y'].mean() * 100:.1f}%")
print(f"y=1 rate in FLAGGED clean rows: {clean.loc[flagged_clean, 'y'].mean() * 100:.1f}%")
print(f"y=1 rate in KEPT clean rows:    {clean.loc[~flagged_clean, 'y'].mean() * 100:.1f}%")

y=1 rate in ALL clean rows:     11.3%
y=1 rate in FLAGGED clean rows: 36.3%
y=1 rate in KEPT clean rows:    5.7%


In [7]:
for ds, target in [("online_shoppers", "Revenue"), ("credit_card", "default.payment.next.month")]:
    data = pd.read_csv(f"../data/messy/{ds}/clean/train.csv")
    cols = [c for c in data.select_dtypes(include="number").columns if c != target]
    flagged = IsolationForest(contamination="auto", random_state=42).fit(data[cols]).predict(data[cols]) == -1
    print(f"{ds}: flagged {flagged.mean()*100:.1f}% of clean rows | "
          f"positive rate all {data[target].mean()*100:.1f}% vs flagged {data.loc[flagged, target].mean()*100:.1f}%")

online_shoppers: flagged 7.6% of clean rows | positive rate all 15.5% vs flagged 28.6%
credit_card: flagged 6.8% of clean rows | positive rate all 22.1% vs flagged 26.6%


In [8]:
clean = pd.read_csv("../data/messy/bank_marketing/clean/train.csv")
num_cols = ["age", "duration", "campaign", "pdays", "previous",
            "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

flagged = pd.Series(False, index=clean.index)
for col in num_cols:
    q1 = clean[col].quantile(0.25)
    q3 = clean[col].quantile(0.75)
    iqr = q3 - q1
    outside = (clean[col] < q1 - 1.5 * iqr) | (clean[col] > q3 + 1.5 * iqr)
    flagged = flagged | outside

print(f"clean rows flagged by IQR: {flagged.mean() * 100:.1f}%")
print(f"y=1 rate in flagged rows:  {clean.loc[flagged, 'y'].mean() * 100:.1f}%  (normal: {clean['y'].mean() * 100:.1f}%)")

clean rows flagged by IQR: 26.3%
y=1 rate in flagged rows:  27.4%  (normal: 11.3%)
